# Car Fuel Efficiency: Tree-Based Regression

Comparing decision trees, random forests, and XGBoost for predicting car fuel efficiency.

Dataset: `data/car_fuel_efficiency_2026.csv`

Target: `fuel_efficiency_mpg`

## Setup

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [2]:
df = pd.read_csv("data/car_fuel_efficiency_2026.csv")
df = df.fillna(0)
df.shape

(10000, 11)

In [3]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [4]:
target = "fuel_efficiency_mpg"

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

In [5]:
features = [c for c in df_train.columns if c != target]

train_dicts = df_train[features].to_dict(orient="records")
val_dicts = df_val[features].to_dict(orient="records")
test_dicts = df_test[features].to_dict(orient="records")

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

X_train.shape, X_val.shape, X_test.shape

((6000, 16), (2000, 16), (2000, 16))

## Decision Tree Split

Fit a depth-1 decision tree to identify the single most informative feature for predicting fuel efficiency.

In [6]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()
importances = pd.Series(dt.feature_importances_, index=feature_names)
importances[importances > 0].sort_values(ascending=False)

model_year    1.0
dtype: float64

In [7]:
split_feature = importances.idxmax()
original_feature = split_feature.split("=")[0]

print("one-hot feature:", split_feature)
print("original feature:", original_feature)

one-hot feature: model_year
original feature: model_year


**Result:** the depth-1 tree splits on `model_year` at a threshold of ~1997.5.

## Random Forest Baseline

Fit a random forest with `n_estimators=10`, `random_state=1`, and `n_jobs=-1`, then measure its validation RMSE.

In [8]:
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
rmse

1.8332197358745623

**Result:** the validation RMSE is approximately 1.833.

## Number of Trees

Compare `n_estimators` values `[10, 50, 100, 150]` and identify the one with the lowest validation RMSE.

In [9]:
scores = {}

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)

    score = root_mean_squared_error(y_val, rf.predict(X_val))
    scores[n] = round(score, 3)
    print(f"n_estimators={n:>3}: rmse={score:.3f}")

n_estimators= 10: rmse=1.833


n_estimators= 50: rmse=1.774


n_estimators=100: rmse=1.770


n_estimators=150: rmse=1.769


In [10]:
min(scores, key=scores.get)

150

**Result:** 150 trees gives the lowest validation RMSE (1.769).

## Tree Depth

For each `max_depth` in `[10, 15, 20, 25]`, average the validation RMSE across `n_estimators` in `[10, 50, 100, 150]` to select the best depth.

In [11]:
mean_scores = {}

for depth in [10, 15, 20, 25]:
    scores = []

    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1,
        )
        rf.fit(X_train, y_train)
        scores.append(root_mean_squared_error(y_val, rf.predict(X_val)))

    mean_scores[depth] = np.mean(scores)
    print(f"max_depth={depth:>2}: mean rmse={mean_scores[depth]:.3f}")

max_depth=10: mean rmse=1.754


max_depth=15: mean rmse=1.784


max_depth=20: mean rmse=1.787


max_depth=25: mean rmse=1.786


In [12]:
min(mean_scores, key=mean_scores.get)

10

**Result:** `max_depth=10` gives the lowest mean validation RMSE (~1.754).

## Feature Importance

Fit a random forest with `n_estimators=10`, `max_depth=20`, `random_state=1`, and `n_jobs=-1` to rank feature importance.

In [13]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

importances = pd.Series(
    rf.feature_importances_,
    index=dv.get_feature_names_out(),
)
importances.sort_values(ascending=False).head(10)

model_year                      0.296107
vehicle_weight                  0.203313
fuel_type=Gasoline              0.160712
horsepower                      0.078271
engine_displacement             0.061281
acceleration                    0.054998
origin=USA                      0.042152
drivetrain=Front-wheel drive    0.020062
num_doors                       0.019401
fuel_type=Diesel                0.016571
dtype: float64

In [14]:
by_feature = importances.groupby(
    lambda name: name.split("=")[0]
).sum()

by_feature.sort_values(ascending=False)

model_year             0.296107
vehicle_weight         0.203313
fuel_type              0.193555
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
origin                 0.052103
drivetrain             0.033651
num_doors              0.019401
num_cylinders          0.007318
dtype: float64

**Result:** `vehicle_weight` is the most important of the four features, with importance ~0.203.

## XGBoost Learning Rate

Train XGBoost for 100 rounds and compare the validation RMSE for `eta=0.3` and `eta=0.1`.

In [15]:
feature_names = dv.get_feature_names_out().tolist()

dtr = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dva = xgb.DMatrix(X_val, label=y_val, feature_names=feature_names)

xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

watchlist = [(dtr, "train"), (dva, "val")]

In [16]:
results = {}

for eta in [0.3, 0.1]:
    params = {**xgb_params, "eta": eta}
    model = xgb.train(
        params,
        dtr,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False,
    )

    rmse = root_mean_squared_error(y_val, model.predict(dva))
    results[eta] = rmse
    print(f"eta={eta}: final val rmse={rmse:.4f}")

eta=0.3: final val rmse=1.8265


eta=0.1: final val rmse=1.7248


In [17]:
min(results, key=results.get)

0.1

**Result:** `eta=0.1` gives the lower validation RMSE (1.725 vs 1.827).